In [87]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

## Whenever your team wants to pass a patient data for the first time, they will take 5 values of each category, Heart Rate, Dias BP, Sys BP and SPO2 and preprocess it similar to this. The final initial_array will have 20 values, i.e., 5 for each category initially.

## It is your choice to normalize the data. If you want to normalize the data, then you can divide all values by the maximum value in the dataset. That will scale those values between 0 and 1. It might not be necessary in this case since the values are not far apart

In [79]:
heartbeats = [json.loads(line) for line in open('Rob_Fisher_HR.json', 'r')]
bloodpressures = [json.loads(line) for line in open('Rob_Fisher_BP.json', 'r')]
spo2 = [json.loads(line) for line in open('Rob_Fisher_SPO2.json', 'r')]
    
HRs = []
dias_BPs = []
sys_BPs = []
SPO2s = []

for idx, data in enumerate(heartbeats):
    if idx == 5:
        break
    HRs.append(data['heartRate'])
    
for idx, data in enumerate(bloodpressures):
    if idx == 5:
        break
    dias_BPs.append(data['bloodPressureDiastolic'])
    sys_BPs.append(data['bloodPressureSystolic'])
    
for idx, data in enumerate(spo2):
    if idx == 5:
        break
    SPO2s.append(data['spO2'])
    
print(HRs, dias_BPs, sys_BPs, SPO2s)

initial_array = []

for x in HRs:
    initial_array.append(x)
    
for x in dias_BPs:
    initial_array.append(x)
    
for x in sys_BPs:
    initial_array.append(x)
    
for x in SPO2s:
    initial_array.append(x)
    
initial_array

[65, 58, 82, 94, 78] [72, 71, 60, 69, 64] [122, 121, 110, 119, 114] [97, 97, 98, 97, 98]


[65,
 58,
 82,
 94,
 78,
 72,
 71,
 60,
 69,
 64,
 122,
 121,
 110,
 119,
 114,
 97,
 97,
 98,
 97,
 98]

In [80]:
class Model(nn.Module):
    def __init__(self, device):
        super(Model, self).__init__()
        self.device = device
        self.lstm_1 = nn.LSTM(5, 10, batch_first = True, bidirectional = True)
        self.lstm_2 = nn.LSTM(20, 10, batch_first = True, bidirectional = True)
        self.linear_1 = nn.Linear(4 * 20, 16)
        self.linear_2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, X):
        c_0, h_0 = torch.randn((2, X.shape[0], 10)).to(self.device), torch.randn((2, X.shape[0], 10)).to(self.device)
        c_1, h_1 = torch.randn((2, X.shape[0], 10)).to(self.device), torch.randn((2, X.shape[0], 10)).to(self.device)
        Y, (_, _) = self.lstm_1(X, (c_0, h_0))
        Y, (_, _) = self.lstm_2(Y, (c_1, h_1))
        Y = torch.flatten(Y, start_dim = 1)
        Y = self.linear_1(Y)
        Y = self.linear_2(Y)
        Y = self.sigmoid(Y)
        return Y

## The Deep Learning Model using LSTMs is created above. In the below function, once the model has been trained, pass first = True for the first time when you are passing the patient's data. Remember the data needs to be in a set of 5, i.e., 5 Heart Rates, 5 Diastolic BPs, 5 Systolic BPs, 5 SPO2 Level. Afterwards, whenever you want to pass additional data, set first = False for that patient and whatever value you want to pass, set that value and set all other values as None. So if you want to pass additional heartbeat, then set new_heartbeat = your_value and keep all other None. So, only for the first time, you would need 5 values of each. Afterwards, you can keep on passing single values per function call. Set device = 'cuda' if you have a GPU with cuda-enabled else set device = 'cpu'

## The function will return the prediction of being abnormal or normal, also the probability of its prediction and the initial array with the modified value. So if initially you pass the initial array, you will get the same array in return. Afterwards, if you pass a new heart rate value, you will get the modified initial array with the new value included. So, each time you don't have to create a new initial array, you will just pass the returned one into the function. When the model is trained, set the model_weights_path to the path containing the model's trained weights. The weights will be loaded and the prediction will be made

In [81]:
def algorithm(initial_array, model_weights_path, first = True, new_heartbeat = None, new_diasbp = None, new_sysbp = None, new_spo2 = None, device = 'cpu'):
    if not first:
        if new_heartbeat is not None:
            for i in range(4):
                initial_array[i] = initial_array[i + 1]
            initial_array[4] = new_heartbeat
        if new_diasbp is not None:
            for i in range(5, 9):
                initial_array[i] = initial_array[i + 1]
            initial_array[9] = new_diasbp
        if new_sysbp is not None:
            for i in range(10, 14):
                initial_array[i] = initial_array[i + 1]
            initial_array[14] = new_sysbp
        if new_spo2 is not None:
            for i in range(15, 19):
                initial_array[i] = initial_array[i + 1]
            initial_array[19] = new_spo2
    
    X = torch.Tensor(initial_array).reshape(1, 4, 5).to(device)
    
    model = Model(device).to(device)
    model.load_state_dict(torch.load(model_weights_path), map_location = device)
    Y = model(X)
    
    if Y >= 0.5:
        prediction = 'Abnormal'
    else:
        prediction = 'Normal'
        
    return prediction, Y[0][0].detach().cpu().numpy(), initial_array

In [84]:
prediction, probability, initial_array = algorithm(initial_array, model_weights_path = '', first = False, new_heartbeat = None, new_diasbp = None, new_sysbp = None, new_spo2 = 80, device = 'cuda')

In [85]:
prediction, probability, initial_array

('Normal',
 array(0.48021406, dtype=float32),
 [58,
  82,
  94,
  78,
  35,
  71,
  60,
  69,
  64,
  55,
  122,
  121,
  110,
  119,
  114,
  97,
  98,
  97,
  98,
  80])

## I am writing a custom training loop for PyTorch here for you to train the model. For training purposes, you will create a PyTorch dataloader. Your team will know how to create one, if not, then it can be looked on the PyTorch website. Additionally, for training, you will pass many patients' data in sets of 5. So if you have HRs, Dias BPs, Sys BPs and SPO2s of 5 patients then take first 5 values of each so you will have 20 values per patient. This will give you 20 values for 5 patients. An array of shape (5, 20) which will be reshaped into (5, 4, 5) in the subsequent function. For each set of 20 values, you will label that as Abnormal or Normal. So for first patient (1, 20) will be labelled as Normal or Abnormal and so on. If you want more values, then divide them into sets of 5. So for example, if you have 10 HRs, 10 Dias BPs, 10 Sys BPs and 10 SPO2 for a patient then divide them into 2 intervals and thus the resulting array would become (2, 20). The same patient but 2 times. First time, first 5, 5, 5, 5, i.e. first 20 values then the same patient but next 20 values. Afterwards, the final input array would be of shape (total_patients, 20) and the labels would be (total_patients, 1). Where 0 is 'Normal' and 1 is 'Abnormal'. So for each set of 20 values, a single digit 0 or 1, i.e. Normal or Abnormal

In [ ]:
def train(dataloader, test_dataloader, learning_rate, device, epochs, display, continue_training = False):
    model = Model(device).to(device)
    optim = torch.optim.Adam(model.params(), lr = learning_rate)
    
    if continue_training:
        model.load_state_dict(torch.load('Disease_Model.pth'))
        optim.load_state_dict(torch.load('Disease_Optimizer.pth'))
        
    criterion = nn.BCELoss()
    
    losses = []
    
    train_accuracies = []
    
    for epoch in range(epochs):
        
        accuracies = []
        test_accuracies = []
        
        for data in tqdm(dataloader):
            X, Y = data
            X = X.reshape(len(X), 4, 5)
            X = X.to(device)
            Y = Y.to(device)
            Y = Y.float()
            
            optim.zero_grad()
            
            Y_h = model(X)
            
            loss = criterion(Y_h, Y)
            losses += [loss.item()]
            loss.backward()
            optim.step()
            
            accuracies.append(accuracy_score(Y.cpu(), np.where(Y_h.cpu() >= 0.5, 1, 0)))
            train_accuracies.append(accuracy_score(Y.cpu(), np.where(Y_h.cpu() >= 0.5, 1, 0)))
            
        if (epoch + 1) % display == 0:
            for data in tqdm(test_dataloader):
                X, Y = data
                X = X.reshape(len(X), 4, 5)
                X = X.to(device)
                Y = Y.to(device)
                Y = Y.float()
                
                Y_h = model(X)
                
                test_accuracies.append(accuracy_score(Y.cpu(), np.where(Y_h.cpu() >= 0.5, 1, 0)))
            
            print('At epoch', epoch + 1, 'loss:', np.average(losses), 'train accuracy:', np.average(accuracies), 'test accuracy', np.average(test_accuracies))
            step_bins = 20
            length = len(losses)
            num_examples = (length // step_bins) * step_bins
            plt.plot(
                range(num_examples // step_bins),
                torch.Tensor(losses[ : num_examples]).view(-1, step_bins). mean(1),
                label = 'Model Loss'
            )
            plt.plot(
                range(num_examples // step_bins),
                torch.Tensor(train_accuracies[ : num_examples]).view(-1, step_bins).mean(1),
                label = 'Training Accuracy'
            )
            plt.legend()
            plt.show()
            
        torch.save(model.state_dict(), 'Disease_Model.pth')
        torch.save(optim.state_dict(), 'Disease_Optimizer.pth')

In [88]:
learning_rate = 0.001
device = 'cuda'
epochs = 100
display = 200 # After how many steps to display the losses and accuracy of the model

# train_dataloader = ...
# test_dataloader = ...

In [ ]:
train(train_dataloader, test_dataloader, learning_rate, device, epochs, display, continue_training = False)